# Consolidated Tables & LaTeX Analysis Pipeline

Unified pipeline to discover, filter, and export LaTeX tables from `Plots/**/Tables/*.csv`:
- Automatically extracts `ModelsCompared` (e.g. `Model_V10_100_Rings_vs_Athena_Cut_Loose`), `ComparisonRun`, `ET`, and `ETA`.
- Each table group follows the same workflow: **Load -> Filter -> Format & Export LaTeX**.

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import sys
from pathlib import Path
from typing import Optional

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.core.Validation.DistanceMetricsETL import DistanceMetricsETL
from src.core.Validation.LatexTableFormatter import LatexTableFormatter

print(f"Project root: {PROJECT_ROOT}")

## 1. Folder Configuration & Shared Helpers

In [ ]:
PLOTS_ROOT: Path = PROJECT_ROOT / "Plots"
INPUT_FOLDERS: list[Path] = [
    PLOTS_ROOT,
    # PROJECT_ROOT / "Plots/Model_V10_100_Rings_vs_Athena_Cut_Loose",
    # PROJECT_ROOT / "Plots/Model_V10_100_Rings_vs_Model_V10_50_Rings",
]

OUTPUT_DIR: Path = PLOTS_ROOT / "ConsolidatedAnalysis"
TABLES_DIR: Path = OUTPUT_DIR / "Tables"
LATEX_DIR: Path = OUTPUT_DIR / "LaTeX"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
LATEX_DIR.mkdir(parents=True, exist_ok=True)

def _parse_models_and_run(csv_path: Path, plots_root: Path = PLOTS_ROOT) -> tuple[str, str]:
    """Extracts (ModelsCompared, ComparisonRun) from Plots/<ModelsCompared>/<ComparisonRun>/Tables/<file>.csv."""
    try:
        rel = csv_path.relative_to(plots_root).parts
    except ValueError:
        rel = csv_path.parts
    if len(rel) >= 4 and rel[-2] == "Tables":
        return rel[-4], rel[-3]
    if len(rel) >= 3 and rel[-2] == "Tables":
        return rel[-3], rel[-3]
    return csv_path.parent.name, csv_path.parent.name

def _parse_et_eta(text: str) -> tuple[Optional[int], Optional[int]]:
    m = re.search(r"et_?(\d+).*?eta_?(\d+)", text)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def _discover_csvs_in_folders(folders: list[Path], file_pattern: str) -> list[Path]:
    """Discovers matching CSVs across a list of base, model-pair, run, or Tables folders."""
    seen: set[Path] = set()
    discovered: list[Path] = []
    for folder in folders:
        folder = Path(folder)
        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue
        matches = list(folder.glob(file_pattern)) if folder.name == "Tables" else list(folder.glob(f"**/Tables/{file_pattern}"))
        for p in sorted(matches):
            resolved = p.resolve()
            if not p.name.startswith(".") and not any(part.startswith(".sys.v#") for part in p.parts) and "ConsolidatedAnalysis" not in resolved.parts and resolved not in seen:
                seen.add(resolved)
                discovered.append(p)
    return sorted(discovered)

def load_table_group(file_pattern: str, folders: list[Path] = INPUT_FOLDERS, plots_root: Path = PLOTS_ROOT) -> pd.DataFrame:
    """Discovers and loads all CSVs matching `file_pattern` across `folders`, adding ModelsCompared."""
    csv_files = _discover_csvs_in_folders(folders, file_pattern)
    print(f"[{file_pattern}] Discovered {len(csv_files)} files across {len(folders)} folder(s)")
    frames = []
    for p in csv_files:
        part = pd.read_csv(p)
        models_compared, comp_run = _parse_models_and_run(p, plots_root)
        et, eta = _parse_et_eta(p.name)
        part.insert(0, "ModelsCompared", models_compared)
        part.insert(1, "ComparisonRun", comp_run)
        part.insert(2, "ET", et)
        part.insert(3, "ETA", eta)
        part["SourceFile"] = p.name
        frames.append(part)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def apply_common_filters(
    df: pd.DataFrame,
    models_filter: Optional[list[str]] = None,
    run_contains: Optional[str] = None,
    et_filter: Optional[list[int]] = None,
    eta_filter: Optional[list[int]] = None,
) -> pd.DataFrame:
    """Applies standard ModelsCompared, ComparisonRun, ET, and ETA filters."""
    out = df.copy()
    if models_filter:
        out = out[out["ModelsCompared"].isin(models_filter)]
    if run_contains:
        out = out[out["ComparisonRun"].str.contains(run_contains, na=False)]
    if et_filter:
        out = out[out["ET"].isin(et_filter)]
    if eta_filter:
        out = out[out["ETA"].isin(eta_filter)]
    return out.reset_index(drop=True)

def save_group_outputs(df_raw: pd.DataFrame, df_latex: pd.DataFrame, group_name: str, caption: str, label: str) -> str:
    """Saves consolidated CSV and formatted LaTeX table under Plots/ConsolidatedAnalysis/."""
    csv_path = TABLES_DIR / f"consolidated_{group_name}.csv"
    tex_path = LATEX_DIR / f"consolidated_{group_name}_latex_table.tex"
    df_raw.to_csv(csv_path, index=False)
    latex_str = LatexTableFormatter.to_latex(df_latex, caption=caption, label=label)
    tex_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved CSV:   {csv_path}")
    print(f"Saved LaTeX: {tex_path}")
    return latex_str

available_runs = sorted({_parse_models_and_run(p, PLOTS_ROOT)[0] for p in _discover_csvs_in_folders(INPUT_FOLDERS, "*.csv")})
print("Available model comparison folders:", available_runs)

## 2. Group 1 — Variable JS Distance (`*_variable_js_divergence_*.csv`)

In [ ]:
# 2a. Load
df_js = load_table_group("*_variable_js_divergence_*.csv")
display(df_js.head(10))

In [ ]:
# 2b. Filter
JS_MODELS_FILTER: Optional[list[str]] = None     # e.g. ["Model_V10_100_Rings_vs_Athena_Cut_Loose"]
JS_RUN_CONTAINS: Optional[str] = None            # e.g. "best_sp_vs_athena_cut"
JS_FEATURES_FILTER: Optional[list[str]] = ["Eratio", "trig_L2_calo_et", "trig_L2_calo_weta2"]
JS_REFERENCE_FILTER: Optional[str] = None        # e.g. "True Signal" or "True Background"

df_js_filt = apply_common_filters(df_js, models_filter=JS_MODELS_FILTER, run_contains=JS_RUN_CONTAINS)
if not df_js_filt.empty:
    if JS_FEATURES_FILTER and "Feature" in df_js_filt.columns:
        df_js_filt = df_js_filt[df_js_filt["Feature"].isin(JS_FEATURES_FILTER)]
    if JS_REFERENCE_FILTER and "Reference" in df_js_filt.columns:
        df_js_filt = df_js_filt[df_js_filt["Reference"] == JS_REFERENCE_FILTER]
    df_js_filt = DistanceMetricsETL.evaluate_distribution_equivalence(df_js_filt)
display(df_js_filt)

In [ ]:
# 2c. Format & Export LaTeX
if not df_js_filt.empty:
    js_col = "JS Distance (sqrt(D_JS))" if "JS Distance (sqrt(D_JS))" in df_js_filt.columns else "JS Divergence"
    js_unc_col = "JS Distance (sqrt(D_JS)) Uncertainty" if "JS Distance (sqrt(D_JS)) Uncertainty" in df_js_filt.columns else "JS Uncertainty"
    js_keep = ["ModelsCompared", "ET", "ETA", "Feature", "Comparison", js_col, js_unc_col]
    js_sub = df_js_filt[[c for c in js_keep if c in df_js_filt.columns]]
    js_fmt = LatexTableFormatter.merge_uncertainty_pairs(
        js_sub,
        pairs={"JS Distance ($\\sqrt{D_{\\text{JS}}}$)": (js_col, js_unc_col)},
        sig_figs=2,
        drop_original=True,
    )
    display(js_fmt)
    js_tex = save_group_outputs(
        df_js_filt, js_fmt, "variable_js_distance",
        caption="Variable distribution JS distance ($\\sqrt{D_{\\text{JS}}}$) across comparisons.",
        label="tab:variable_js_distance",
    )
    print(js_tex)

## 3. Group 2 — Quadrant Strategy Metrics (`*_quadrant_strategy_metrics.csv`)

In [ ]:
# 3a. Load
df_strat = load_table_group("*_quadrant_strategy_metrics.csv")
display(df_strat.head(10))

In [ ]:
# 3b. Filter
STRAT_MODELS_FILTER: Optional[list[str]] = None
STRAT_RUN_CONTAINS: Optional[str] = None

df_strat_filt = apply_common_filters(df_strat, models_filter=STRAT_MODELS_FILTER, run_contains=STRAT_RUN_CONTAINS)
display(df_strat_filt)

In [ ]:
# 3c. Format & Export LaTeX
if not df_strat_filt.empty:
    df_strat_pct = df_strat_filt.copy()
    for col in ["PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"]:
        if col in df_strat_pct.columns:
            df_strat_pct[col] = df_strat_pct[col] * 100.0
    strat_keep = ["ModelsCompared", "ComparisonRun", "Strategy", "PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"]
    strat_sub = df_strat_pct[[c for c in strat_keep if c in df_strat_pct.columns]]
    strat_fmt = LatexTableFormatter.merge_uncertainty_pairs(
        strat_sub,
        pairs={"PD [\%]": ("PD", "PD_unc"), "PF [\%]": ("PF", "PF_unc"), "SP [\%]": ("SP", "SP_unc")},
        precision=2,
        drop_original=True,
    )
    display(strat_fmt)
    strat_tex = save_group_outputs(
        df_strat_filt, strat_fmt, "quadrant_strategy_metrics",
        caption="Quadrant strategy performance metrics ($P_D$, $P_F$, $SP$).",
        label="tab:quadrant_strategy_metrics",
    )
    print(strat_tex)

## 4. Group 3 — Quadrant Counts (`*_quadrant_counts.csv`)

In [ ]:
# 4a. Load
df_counts = load_table_group("*_quadrant_counts.csv")
display(df_counts.head(10))

In [ ]:
# 4b. Filter
COUNTS_MODELS_FILTER: Optional[list[str]] = None
COUNTS_RUN_CONTAINS: Optional[str] = None

df_counts_filt = apply_common_filters(df_counts, models_filter=COUNTS_MODELS_FILTER, run_contains=COUNTS_RUN_CONTAINS)
display(df_counts_filt)

In [ ]:
# 4c. Format & Export LaTeX
if not df_counts_filt.empty:
    counts_keep = ["ModelsCompared", "ComparisonRun", "Quadrant", "Count", "Ratio"]
    counts_fmt = df_counts_filt[[c for c in counts_keep if c in df_counts_filt.columns]]
    display(counts_fmt)
    counts_tex = save_group_outputs(
        df_counts_filt, counts_fmt, "quadrant_counts",
        caption="Quadrant agreement counts and ratios across comparisons.",
        label="tab:quadrant_counts",
    )
    print(counts_tex)

## 5. Group 4 — Pileup Fit & Efficiency vs $\langle\mu\rangle$ (`*_pileup_*.csv`)

In [ ]:
# 5a. Load
df_pileup_fit = load_table_group("*_pileup_fit_parameters.csv")
df_pileup_eff = load_table_group("*_pileup_efficiency_vs_mu_*.csv")
display(df_pileup_fit)
display(df_pileup_eff.head(10))

In [ ]:
# 5b. Filter
PILEUP_MODELS_FILTER: Optional[list[str]] = None
PILEUP_RUN_CONTAINS: Optional[str] = None

df_pileup_fit_filt = apply_common_filters(df_pileup_fit, models_filter=PILEUP_MODELS_FILTER, run_contains=PILEUP_RUN_CONTAINS)
df_pileup_eff_filt = apply_common_filters(df_pileup_eff, models_filter=PILEUP_MODELS_FILTER, run_contains=PILEUP_RUN_CONTAINS)
display(df_pileup_fit_filt)

In [ ]:
# 5c. Format & Export LaTeX
if not df_pileup_fit_filt.empty:
    fit_keep = ["ModelsCompared", "ET", "ETA", "Strategy", "Target PD (%)", "Slope (a)", "Intercept (b)", "R^2", "Fit Formula"]
    fit_fmt = df_pileup_fit_filt[[c for c in fit_keep if c in df_pileup_fit_filt.columns]]
    display(fit_fmt)
    fit_tex = save_group_outputs(
        df_pileup_fit_filt, fit_fmt, "pileup_fit_parameters",
        caption="Linear pileup threshold calibration parameters $T(\\mu) = a\\mu + b$.",
        label="tab:pileup_fit_parameters",
    )
    print(fit_tex)